# Text Sammury

In [ ]:
import pandas as pd

df = pd.read_csv("ayudas_25-26.csv")

display(df)

,Academic_Year,Issuing_Body,BOE_Publication_Date,Deadline_University,Deadline_NonUniversity,Late_Application_Allowed,Application_Channel,Total_Budget,Fixed_Income_Amount,Residence_Amount,Minimum_Variable_Amount,Eligible_Studies_NonUniversity,Eligible_Studies_University,Explicit_Exclusions
0,2025-2026,"Ministerio de Educación, Formación Profesional...",15 de junio,30 de septiembre de 2025,24 de marzo de 2025,31 de diciembre de 2025,vía\nSMS,concesión directa y\npago de una beca,"1.700,00 euros","2.700,00 euros",beca de matrícula,Artículo 63,arquitectura e\ningeniería,"créditos convalidados, reconocidos o adaptados"


In [ ]:
import re

def normalize_row(row):
    out = dict(row)

    # Core metadata
    out["Academic_Year"] = row.get("Academic_Year")
    out["Issuing_Body"] = row.get("Issuing_Body")
    out["BOE_Publication_Date"] = row.get("BOE_Publication_Date")

    # Deadlines / procedure
    out["Deadline_University"] = row.get("Deadline_University")
    out["Deadline_NonUniversity"] = row.get("Deadline_NonUniversity")
    out["Late_Application_Allowed"] = row.get("Late_Application_Allowed")
    out["Application_Channel"] = row.get("Application_Channel")

    # Amounts / budget
    out["Total_Budget"] = row.get("Total_Budget")
    out["Fixed_Income_Amount"] = row.get("Fixed_Income_Amount")
    out["Residence_Amount"] = row.get("Residence_Amount")
    out["Minimum_Variable_Amount"] = row.get("Minimum_Variable_Amount")

    # Scope of studies
    out["Eligible_Studies_NonUniversity"] = row.get("Eligible_Studies_NonUniversity")
    out["Eligible_Studies_University"] = row.get("Eligible_Studies_University")
    out["Explicit_Exclusions"] = row.get("Explicit_Exclusions")

    return out

In [ ]:
def summary_from_row(r):
    year = r.get("Academic_Year") or "—"
    body = r.get("Issuing_Body") or ""
    pub = r.get("BOE_Publication_Date") or ""

    du = r.get("Deadline_University") or ""
    dnu = r.get("Deadline_NonUniversity") or ""
    late = r.get("Late_Application_Allowed") or ""
    channel = r.get("Application_Channel") or ""

    renta = r.get("Fixed_Income_Amount") or ""
    resi = r.get("Residence_Amount") or ""
    varmin = r.get("Minimum_Variable_Amount") or ""
    budget = r.get("Total_Budget") or ""

    nonuni = r.get("Eligible_Studies_NonUniversity") or ""
    uni = r.get("Eligible_Studies_University") or ""
    excl = r.get("Explicit_Exclusions") or ""

    parts = []

    header = f"La convocatoria de becas del curso académico {year}"
    if pub:
        header += f" (publicada en BOE el {pub})"
    header += "."
    parts.append(header)

    if body:
        parts.append(f"Organismo convocante: {body}.")

    # Deadlines
    if du or dnu:
        if du and dnu:
            parts.append(f"Plazos de solicitud: universitarios hasta {du}; no universitarios hasta {dnu}.")
        elif du:
            parts.append(f"Plazo de solicitud (universitarios): hasta {du}.")
        else:
            parts.append(f"Plazo de solicitud (no universitarios): hasta {dnu}.")
    else:
        parts.append("En los datos extraídos no consta un plazo de solicitud explícito.")

    # Late application
    if late:
        parts.append(f"Presentación fuera de plazo / supuestos excepcionales: {late}.")

    # Application channel / procedure
    if channel:
        parts.append(f"Canal y procedimiento de solicitud: {channel}.")

    # Amounts
    amount_bits = []
    if renta:
        amount_bits.append(f"cuantía fija ligada a la renta: {renta}")
    if resi:
        amount_bits.append(f"cuantía fija ligada a la residencia: {resi}")
    if varmin:
        amount_bits.append(f"cuantía variable mínima: {varmin}")
    if budget:
        amount_bits.append(f"presupuesto/financiación máxima: {budget}")

    if amount_bits:
        parts.append("Importes: " + "; ".join(amount_bits) + ".")
    else:
        parts.append("En los datos extraídos no constan importes detallados de la convocatoria.")

    # Eligible studies / exclusions
    if nonuni:
        parts.append(f"Enseñanzas no universitarias incluidas: {nonuni}.")
    if uni:
        parts.append(f"Estudios universitarios incluidos: {uni}.")
    if excl:
        parts.append(f"Exclusiones explícitas: {excl}.")

    return " ".join(parts)


# -----------------------------
# Corpus summary
# -----------------------------
def make_template_summary(df):
    df2 = df.copy()

    # Normalize rows
    rows = [normalize_row(row) for _, row in df2.iterrows()]

    out = []
    out.append("Resumen del corpus de convocatorias de becas (BOE) a partir de la información extraída en CSV.\n")

    for r in rows:
        out.append(summary_from_row(r))
        out.append("")  # blank line

    return "\n".join(out).strip()

In [ ]:
template_summary = make_template_summary(df)
print(template_summary)

Resumen del corpus de convocatorias de becas (BOE) a partir de la información extraída en CSV.

La convocatoria de becas del curso académico 2025-2026 (publicada en BOE el 15 de junio). Organismo convocante: Ministerio de Educación, Formación Profesional y Deportes. Plazos de solicitud: universitarios hasta 30 de septiembre de 2025; no universitarios hasta 24 de marzo de 2025. Presentación fuera de plazo / supuestos excepcionales: 31 de diciembre de 2025. Canal y procedimiento de solicitud: vía
SMS. Importes: cuantía fija ligada a la renta: 1.700,00 euros; cuantía fija ligada a la residencia: 2.700,00 euros; cuantía variable mínima: beca de matrícula; presupuesto/financiación máxima: concesión directa y
pago de una beca. Enseñanzas no universitarias incluidas: Artículo 63. Estudios universitarios incluidos: arquitectura e
ingeniería. Exclusiones explícitas: créditos convalidados, reconocidos o adaptados.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

gen = pipeline("text-generation", model=model, tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
import torch

POLISH_PROMPT = (
    "Reescribe el siguiente contenido en español como un texto claro, natural y bien redactado.\n"
    "El contenido original puede venir de una tabla o de datos estructurados, así que debes convertirlo en un texto corrido y coherente.\n\n"
    "REGLAS ESTRICTAS:\n"
    "- Mantén exactamente el mismo contenido factual.\n"
    "- NO añadas cifras, fechas, importes ni datos que no estén en el texto original.\n"
    "- NO inventes plazos, rangos, causas o conclusiones.\n"
    "- NO repitas información.\n"
    "- NO añadas información extra.\n"
    "- NO añadas párrafos extra innecesarios.\n"
    "- Organiza la información para que se lea como un texto (no como una tabla).\n"
    "- Usa un estilo claro, preciso y conciso.\n\n"
    "TEXTO:\n"
)

def polish_with_local_llm(text, max_new_tokens=500):
    prompt = POLISH_PROMPT + text + "\n\nVERSIÓN MEJORADA:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

polished = polish_with_local_llm(template_summary)
print(polished)

Resumen del corpus de convocatorias de becas (BOE) a partir de la información extraída en CSV.

La convocatoria de becas para el curso académico 2025-2026 (publicada en BOE el 15 de junio) está organizada por el Ministerio de Educación, Formación Profesional y Deportes. Los plazos de solicitud son los siguientes: para los universitarios hasta el 30 de septiembre de 2025; para los no universitarios hasta el 24 de marzo de 2025. Las presentaciones fuera de plazo o con supuestos excepcionales se pueden realizar hasta el 31 de diciembre de 2025. El canal y el procedimiento de solicitud son por SMS. Los importes ofrecidos son los siguientes: una cuantía fija vinculada a la renta de 1.700,00 euros; otra cuantía fija vinculada a la residencia de 2.700,00 euros; y una cuantía variable mínima que incluye la beca de matrícula. La financiación máxima es concesión directa y pago de una beca. Las enseñanzas no universitarias incluidas son las mencionadas en el artículo 63, mientras que las universi